# MGT 599 Capstone - Group 4
## Week 2 Submission

April 2026

This notebook runs the full week 2 pipeline from start to finish. We load the raw data, clean it, explore it, build features, and save the final files for modeling next week. Run the cells top to bottom.

## Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
sys.path.insert(0, str(PROJECT_ROOT))

from common_utils import (
    BASE_PATH, DATA_PATH, CLEANED_PATH,
    TASK1_FEATURE_PATH, TASK2_FEATURE_PATH, MODEL_READY_PATH, FINDINGS_PATH,
    ensure_folders, load_task_dataframe, guess_target_column, guess_text_columns,
    add_basic_text_features, add_keyword_features, safe_text, combine_text_fields,
    simple_missing_summary, find_weak_features, analyze_imbalance, set_chart_style,
    save_dataframe
)

DB_PATH = PROJECT_ROOT / 'data' / 'db' / 'analytics.duckdb'

set_chart_style()
ensure_folders()

print('project root:', PROJECT_ROOT)
print('db path:', DB_PATH)
print('ready')

## Step 1 - load raw data into DuckDB

In [ ]:
from data.loader import DataLoader
from data.validator import DataValidator

loader = DataLoader(str(PROJECT_ROOT))
loader.connect()
loader.load_data()
loader.validate_tables()

validator = DataValidator(loader.con)
validator.validate_task1()
validator.validate_task2()

loader.con.close()
print('step 1 done')

## Step 2 - clean the data and export to CSVs

In [ ]:
from data.cleaner import DataCleaner

cleaner = DataCleaner(str(DB_PATH))
cleaner.connect()
cleaner.clean_task1()
cleaner.clean_task2()
cleaner.export_to_cleaned(str(CLEANED_PATH))
cleaner.con.close()

print('files in data/cleaned/:')
for f in sorted(CLEANED_PATH.glob('*.csv')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

print('step 2 done')

## Step 3 - load cleaned data and look at it

In [ ]:
task1 = load_task_dataframe('task1')
task2 = load_task_dataframe('task2')

print('task1 shape:', task1.shape)
print('task1 columns:', list(task1.columns))
print()
print('task2 shape:', task2.shape)
print('task2 columns:', list(task2.columns))

In [ ]:
task1.head(3)

In [ ]:
task2.head(3)

### missing values

In [ ]:
print('task 1 missing:')
display(simple_missing_summary(task1))

print('task 2 missing:')
display(simple_missing_summary(task2))

### class distribution - industry (task 1)

In [ ]:
t1_target = guess_target_column(task1, 'task1')
t1_counts = task1[t1_target].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

t1_counts.head(15).sort_values().plot(kind='barh', ax=axes[0], color='#2E75B6')
axes[0].set_title('top 15 industry classes - task 1')
axes[0].set_xlabel('count')
for i, v in enumerate(t1_counts.head(15).sort_values().values):
    axes[0].text(v + t1_counts.max() * 0.01, i, str(v), va='center', fontsize=8)

t1_counts.tail(10).sort_values().plot(kind='barh', ax=axes[1], color='#C0504D')
axes[1].set_title('rarest 10 industry classes - task 1')
axes[1].set_xlabel('count')

plt.tight_layout()
plt.show()

t1_imb = analyze_imbalance(task1[t1_target], 'task 1 industry')

### class distribution - subindustry (task 2)

In [ ]:
t2_target = guess_target_column(task2, 'task2')
t2_counts = task2[t2_target].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

t2_counts.head(15).sort_values().plot(kind='barh', ax=axes[0], color='#4BACC6')
axes[0].set_title('top 15 subindustry classes - task 2')
axes[0].set_xlabel('count')

t2_counts.tail(10).sort_values().plot(kind='barh', ax=axes[1], color='#9B59B6')
axes[1].set_title('rarest 10 subindustry classes - task 2')
axes[1].set_xlabel('count')

plt.tight_layout()
plt.show()

t2_imb = analyze_imbalance(task2[t2_target], 'task 2 subindustry')

### text length distributions

In [ ]:
t1_text_cols = guess_text_columns(task1)
t2_text_cols = guess_text_columns(task2)

print('task 1 text columns:', t1_text_cols)
print('task 2 text columns:', t2_text_cols)

fig, axes = plt.subplots(len(t1_text_cols), 1,
                         figsize=(10, 4 * max(len(t1_text_cols), 1)),
                         squeeze=False)

for i, col in enumerate(t1_text_cols):
    lengths = safe_text(task1[col]).apply(len)
    axes[i][0].hist(lengths, bins=50, color='#2E75B6', edgecolor='white')
    axes[i][0].set_title(f'task 1 / {col} - character length')
    axes[i][0].set_xlabel('characters')
    axes[i][0].set_ylabel('count')
    med = lengths.median()
    axes[i][0].axvline(med, color='red', linestyle='--', label=f'median={med:.0f}')
    axes[i][0].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(t2_text_cols), 1,
                         figsize=(10, 4 * max(len(t2_text_cols), 1)),
                         squeeze=False)

for i, col in enumerate(t2_text_cols):
    lengths = safe_text(task2[col]).apply(len)
    axes[i][0].hist(lengths, bins=50, color='#4BACC6', edgecolor='white')
    axes[i][0].set_title(f'task 2 / {col} - character length')
    axes[i][0].set_xlabel('characters')
    axes[i][0].set_ylabel('count')
    med = lengths.median()
    axes[i][0].axvline(med, color='red', linestyle='--', label=f'median={med:.0f}')
    axes[i][0].legend()

plt.tight_layout()
plt.show()

## Step 4 - task 1 feature engineering

We build three types of features: text stats (length, word count etc), keyword flags (0 or 1 per keyword), and TF-IDF (300 numbers per company based on what words appear in the description).

In [ ]:
task1_fe = task1.copy()
task1_fe = add_basic_text_features(task1_fe, t1_text_cols)

t1_main_col_pref = ['LongProfile', 'CompanyDescription', 'SegmentDescription']
t1_main_col = next((c for c in t1_main_col_pref if c in task1_fe.columns),
                   t1_text_cols[0] if t1_text_cols else None)
print('main text column for tfidf:', t1_main_col)

task1_fe['short_desc_flag'] = (
    safe_text(task1_fe[t1_main_col]).str.split().str.len() < 5
).astype(int)
print('short descriptions (<5 words):', task1_fe['short_desc_flag'].sum())

industry_keywords = [
    'bank', 'finance', 'financial', 'insurance', 'retail',
    'software', 'technology', 'healthcare', 'medical', 'pharma',
    'energy', 'oil', 'gas', 'manufacturing', 'industrial',
    'telecom', 'transport', 'logistics',
]
task1_fe = add_keyword_features(task1_fe, t1_main_col, industry_keywords, prefix='industry_kw')
print('keyword features added:', len(industry_keywords))

In [ ]:
print('building tfidf...')
tfidf_vec1 = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X1 = tfidf_vec1.fit_transform(safe_text(task1_fe[t1_main_col]))
tfidf1_df = pd.DataFrame(X1.toarray(),
                          columns=[f'tfidf_{n}' for n in tfidf_vec1.get_feature_names_out()])
print('tfidf shape:', tfidf1_df.shape)

task1_fe = combine_text_fields(task1_fe, t1_text_cols, 'combined_text')
comb_vec1 = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
Xc1 = comb_vec1.fit_transform(safe_text(task1_fe['combined_text']))
comb1_df = pd.DataFrame(Xc1.toarray(),
                         columns=[f'comb_tfidf_{n}' for n in comb_vec1.get_feature_names_out()])
print('combined tfidf shape:', comb1_df.shape)

task1_full = pd.concat([task1_fe.reset_index(drop=True), tfidf1_df.reset_index(drop=True)], axis=1)
print('full task 1 feature file shape:', task1_full.shape)

save_dataframe(task1_fe,   TASK1_FEATURE_PATH / 'task1_features_basic_v1.csv')
save_dataframe(tfidf1_df,  TASK1_FEATURE_PATH / 'task1_tfidf_features_v1.csv')
save_dataframe(comb1_df,   TASK1_FEATURE_PATH / 'task1_combined_tfidf_v1.csv')
save_dataframe(task1_full, TASK1_FEATURE_PATH / 'task1_features_full_v1.csv')
print('task 1 files saved')

In [ ]:
kw_cols1 = [c for c in task1_fe.columns if c.startswith('industry_kw_')]
kw_hits1 = task1_fe[kw_cols1].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
kw_hits1.plot(kind='bar', color='#2E75B6', edgecolor='white')
plt.title('task 1 - keyword hit counts')
plt.ylabel('rows containing keyword')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 5 - task 2 feature engineering

In [ ]:
task2_fe = task2.copy()
task2_fe = add_basic_text_features(task2_fe, t2_text_cols)

t2_main_col_pref = ['SegmentDescription', 'Segment Description', 'SegmentName', 'LongProfile']
t2_main_col = next((c for c in t2_main_col_pref if c in task2_fe.columns),
                   t2_text_cols[0] if t2_text_cols else None)
print('main text column for tfidf:', t2_main_col)

task2_fe['short_desc_flag'] = (
    safe_text(task2_fe[t2_main_col]).str.split().str.len() < 5
).astype(int)
print('short descriptions (<5 words):', task2_fe['short_desc_flag'].sum())

activity_keywords = [
    'delivery', 'transport', 'logistics', 'cloud', 'platform',
    'payments', 'medical', 'device', 'manufacturing', 'processing',
    'software', 'retail', 'distribution', 'construction', 'consulting',
    'analytics', 'security', 'energy', 'packaging', 'warehousing',
    'saas', 'subscription', 'staffing', 'outsourcing', 'brokerage', 'leasing',
]
task2_fe = add_keyword_features(task2_fe, t2_main_col, activity_keywords, prefix='activity_kw')
print('keyword features added:', len(activity_keywords))

In [ ]:
print('building tfidf...')
tfidf_vec2 = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X2 = tfidf_vec2.fit_transform(safe_text(task2_fe[t2_main_col]))
tfidf2_df = pd.DataFrame(X2.toarray(),
                          columns=[f'tfidf_{n}' for n in tfidf_vec2.get_feature_names_out()])
print('tfidf shape:', tfidf2_df.shape)

task2_fe = combine_text_fields(task2_fe, t2_text_cols, 'combined_text')
comb_vec2 = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
Xc2 = comb_vec2.fit_transform(safe_text(task2_fe['combined_text']))
comb2_df = pd.DataFrame(Xc2.toarray(),
                         columns=[f'comb_tfidf_{n}' for n in comb_vec2.get_feature_names_out()])
print('combined tfidf shape:', comb2_df.shape)

task2_full = pd.concat([task2_fe.reset_index(drop=True), tfidf2_df.reset_index(drop=True)], axis=1)
print('full task 2 shape:', task2_full.shape)

save_dataframe(task2_fe,   TASK2_FEATURE_PATH / 'task2_features_basic_v1.csv')
save_dataframe(tfidf2_df,  TASK2_FEATURE_PATH / 'task2_tfidf_features_v1.csv')
save_dataframe(comb2_df,   TASK2_FEATURE_PATH / 'task2_combined_tfidf_v1.csv')
save_dataframe(task2_full, TASK2_FEATURE_PATH / 'task2_features_full_v1.csv')
print('task 2 files saved')

In [ ]:
kw_cols2 = [c for c in task2_fe.columns if c.startswith('activity_kw_')]
kw_hits2 = task2_fe[kw_cols2].sum().sort_values(ascending=False)

plt.figure(figsize=(14, 5))
kw_hits2.plot(kind='bar', color='#4BACC6', edgecolor='white')
plt.title('task 2 - keyword hit counts')
plt.ylabel('rows containing keyword')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 6 - model ready review

Check for columns where almost every row has the same value (not useful for a model), verify row counts, and save the final files.

In [ ]:
weak1 = [c for c, _ in find_weak_features(task1_full)]
weak2 = [c for c, _ in find_weak_features(task2_full)]

print('task 1 near-constant columns:', len(weak1))
print('task 2 near-constant columns:', len(weak2))

task1_ready = task1_full.drop(columns=weak1) if weak1 else task1_full.copy()
task2_ready = task2_full.drop(columns=weak2) if weak2 else task2_full.copy()

print('task 1 final shape:', task1_ready.shape)
print('task 2 final shape:', task2_ready.shape)

In [ ]:
def feature_group_summary(name, df):
    txt   = [c for c in df.columns if any(t in c for t in
             ['_char_len','_word_count','_unique_word','_avg_word','_has_text'])]
    kw    = [c for c in df.columns if '_kw_' in c]
    tf    = [c for c in df.columns if 'tfidf_' in c]
    other = len(df.columns) - len(txt) - len(kw) - len(tf)
    return pd.DataFrame({
        'feature group': ['text stats', 'keyword flags', 'tfidf', 'other/original'],
        'count': [len(txt), len(kw), len(tf), other]
    }).assign(dataset=name)

summary = pd.concat([
    feature_group_summary('task 1', task1_ready),
    feature_group_summary('task 2', task2_ready),
], ignore_index=True)

display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2E75B6', '#4BACC6', '#9B59B6', '#95A5A6']

for i, (dname, grp) in enumerate(summary.groupby('dataset')):
    axes[i].bar(grp['feature group'], grp['count'], color=colors)
    axes[i].set_title(f'{dname} - feature groups')
    axes[i].set_ylabel('number of features')
    axes[i].tick_params(axis='x', rotation=20)
    for j, v in enumerate(grp['count']):
        axes[i].text(j, v + 1, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
save_dataframe(task1_ready, MODEL_READY_PATH / 'task1_model_ready_v1.csv')
save_dataframe(task2_ready, MODEL_READY_PATH / 'task2_model_ready_v1.csv')
print('model-ready files saved')

## Week 2 summary

In [ ]:
print('week 2 results')
print('-' * 40)
print(f'task 1 rows: {task1.shape[0]:,}  |  unique classes: {task1[t1_target].nunique()}')
print(f'task 2 rows: {task2.shape[0]:,}  |  unique classes: {task2[t2_target].nunique()}')
print()
print(f'task 1 top-3 classes cover: {t1_imb["top3_pct"]}% of data')
print(f'task 2 top-3 classes cover: {t2_imb["top3_pct"]}% of data')
print(f'task 1 rare classes (<10 rows): {t1_imb["rare_classes"]}')
print(f'task 2 rare classes (<10 rows): {t2_imb["rare_classes"]}')
print()
print(f'task 1 model-ready file: {task1_ready.shape}')
print(f'task 2 model-ready file: {task2_ready.shape}')
print()
for folder in [TASK1_FEATURE_PATH, TASK2_FEATURE_PATH, MODEL_READY_PATH]:
    csvs = list(folder.glob('*.csv'))
    print(f'{folder.relative_to(PROJECT_ROOT)}: {len(csvs)} csv file(s)')

## What we found and what comes next

The biggest issue we found is class imbalance. Both datasets have a lot of label categories but most of the rows belong to just a few of them. A lot of categories have fewer than 10 examples which is going to be a challenge for the model.

Some descriptions are empty or very short. We flagged those rows with short_desc_flag. The model won't get much signal from a blank description.

The text descriptions do contain useful signals - industry-specific words like lending, semiconductor, cloud appear in the right places. TF-IDF is capturing that.

Next week we train baseline classifiers on the model-ready files (logistic regression and random forest), evaluate with F1-macro score, and see how well the predictions hold up.